In [1]:
import numpy as np
from lava.proc.lif.process import LIF
from lava.proc.dense.process import Dense
from lava.proc.io.sink import RingBuffer as Sink
from lava.magma.core.run_configs import Loihi2SimCfg
from lava.magma.core.run_conditions import RunSteps

# Neuron 0: the "input" neuron - driven by bias to fire regularly
input_neuron = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=5)

# Neuron 1: the "output" neuron - no bias, only fires from synaptic input
output_neuron = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=0)

# Dense connection - this is the synapse
# weights is a matrix: shape (output_size, input_size)
weights = np.array([[15]])  # single synapse, weight = 15

dense = Dense(weights=weights)

# Wire everything together:
# input_neuron -> dense -> output_neuron
input_neuron.s_out.connect(dense.s_in)
dense.a_out.connect(output_neuron.a_in)

# Record both neurons' spikes
sink_input = Sink(shape=(1,), buffer=50)
sink_output = Sink(shape=(1,), buffer=50)

input_neuron.s_out.connect(sink_input.a_in)
output_neuron.s_out.connect(sink_output.a_in)

# Run
input_neuron.run(condition=RunSteps(num_steps=50), 
                 run_cfg=Loihi2SimCfg())

input_spikes = sink_input.data.get().flatten()
output_spikes = sink_output.data.get().flatten()

input_neuron.stop()

print("Input neuron spikes: ", input_spikes[:20])
print("Output neuron spikes:", output_spikes[:20])
print(f"\nTotal input spikes: {np.sum(input_spikes)}")
print(f"Total output spikes: {np.sum(output_spikes)}")

Input neuron spikes:  [0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0.]
Output neuron spikes: [0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

Total input spikes: 16.0
Total output spikes: 47.0


In [2]:
from lava.proc.monitor.process import Monitor

# Rebuild the network fresh
input_neuron = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=5)
output_neuron = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=0)

weights = np.array([[15]])
dense = Dense(weights=weights)

input_neuron.s_out.connect(dense.s_in)
dense.a_out.connect(output_neuron.a_in)

# Probe the output neuron's voltage AND current
monitor = Monitor()
monitor.probe(output_neuron.v, 20)
monitor.probe(output_neuron.u, 20)  # u = synaptic current

input_neuron.run(condition=RunSteps(num_steps=20), 
                 run_cfg=Loihi2SimCfg())

data = monitor.get_data()
input_neuron.stop()

print(data.keys())

AssertionError: Port <lava.magma.core.process.ports.ports.RefPort object at 0x10c7d2980> not found in ProcessModel <class 'lava.proc.monitor.models.PyMonitorModel'>

In [3]:
input_neuron = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=5)
output_neuron = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=0)

weights = np.array([[15]])
dense = Dense(weights=weights)

input_neuron.s_out.connect(dense.s_in)
dense.a_out.connect(output_neuron.a_in)

monitor = Monitor()
monitor.probe(output_neuron.v, 20)

input_neuron.run(condition=RunSteps(num_steps=20), 
                 run_cfg=Loihi2SimCfg())

data = monitor.get_data()
input_neuron.stop()

print(data.keys())

dict_keys(['Process_10'])


In [4]:
v_trace = data['Process_10']['v'].flatten()
print("Output neuron voltage trace:")
print(v_trace)

Output neuron voltage trace:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [5]:
input_neuron2 = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=5)

monitor2 = Monitor()
monitor2.probe(input_neuron2.v, 20)

input_neuron2.run(condition=RunSteps(num_steps=20), 
                  run_cfg=Loihi2SimCfg())

data2 = monitor2.get_data()
input_neuron2.stop()

print(data2.keys())

dict_keys(['Process_13'])


In [6]:
v_trace2 = data2['Process_13']['v'].flatten()
print("Input neuron voltage trace:")
print(v_trace2)

Input neuron voltage trace:
[ 5. 10.  0.  5. 10.  0.  5. 10.  0.  5. 10.  0.  5. 10.  0.  5. 10.  0.
  5. 10.]


In [7]:
input_neuron3 = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=5)
output_neuron3 = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=0)

weights3 = np.array([[3]])  # much smaller weight - can't cross threshold alone
dense3 = Dense(weights=weights3)

input_neuron3.s_out.connect(dense3.s_in)
dense3.a_out.connect(output_neuron3.a_in)

monitor3 = Monitor()
monitor3.probe(output_neuron3.v, 20)

sink_input3 = Sink(shape=(1,), buffer=20)
sink_output3 = Sink(shape=(1,), buffer=20)
input_neuron3.s_out.connect(sink_input3.a_in)
output_neuron3.s_out.connect(sink_output3.a_in)

input_neuron3.run(condition=RunSteps(num_steps=20), 
                  run_cfg=Loihi2SimCfg())

data3 = monitor3.get_data()
input_spikes3 = sink_input3.data.get().flatten()
output_spikes3 = sink_output3.data.get().flatten()
input_neuron3.stop()

print(data3.keys())
print("Input spikes: ", input_spikes3)
print("Output spikes:", output_spikes3)

dict_keys(['Process_16'])
Input spikes:  [0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0.]
Output spikes: [0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1.]


In [8]:
v_trace3 = data3['Process_16']['v'].flatten()
print("Output neuron voltage trace:")
print(v_trace3)

Output neuron voltage trace:
[0. 0. 0. 3. 6. 9. 0. 6. 0. 9. 0. 9. 0. 0. 0. 0. 0. 0. 0. 0.]


In [9]:
input_neuron4 = LIF(shape=(1,), vth=10, dv=0, du=0, bias_mant=5)
output_neuron4 = LIF(shape=(1,), vth=10, dv=0.2, du=0.5, bias_mant=0)  # added decay

weights4 = np.array([[3]])
dense4 = Dense(weights=weights4)

input_neuron4.s_out.connect(dense4.s_in)
dense4.a_out.connect(output_neuron4.a_in)

monitor4 = Monitor()
monitor4.probe(output_neuron4.v, 20)

sink_input4 = Sink(shape=(1,), buffer=20)
sink_output4 = Sink(shape=(1,), buffer=20)
input_neuron4.s_out.connect(sink_input4.a_in)
output_neuron4.s_out.connect(sink_output4.a_in)

input_neuron4.run(condition=RunSteps(num_steps=20), 
                  run_cfg=Loihi2SimCfg())

data4 = monitor4.get_data()
input_spikes4 = sink_input4.data.get().flatten()
output_spikes4 = sink_output4.data.get().flatten()
input_neuron4.stop()

print(data4.keys())
print("Input spikes: ", input_spikes4)
print("Output spikes:", output_spikes4)

dict_keys(['Process_22'])
Input spikes:  [0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0. 1. 0. 0.]
Output spikes: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]


In [10]:
v_trace4 = data4['Process_22']['v'].flatten()
print("Output neuron voltage trace:")
print(v_trace4)

Output neuron voltage trace:
[0.         0.         0.         3.         3.9        3.87
 6.471      6.8643     6.33519    8.490027   8.5029591  7.65783603
 9.5540032  9.35706975 8.34258939 0.         1.7142334  2.22850342
 5.21136108 5.88336804]


## Lava Two-Neuron Network — Observations & Results

### Architecture
Two LIF neurons connected via a `Dense` synapse process:
- Input neuron: driven by constant bias (`bias_mant=5`), fires regularly
- Output neuron: no bias, fires only from synaptic input via the `Dense` connection
- `Dense` process holds the synaptic weight matrix connecting the two

### Initial unexpected result
With `weights=[[15]]` and `du=0` (no current decay) on the output neuron, 
the output fired continuously from step 3 onwards (47 spikes in 50 steps) 
rather than briefly after each input spike as expected.

### Investigation: probing voltage directly
Used Lava's `Monitor` process to probe the output neuron's membrane voltage 
directly. Voltage trace showed all zeros throughout the entire run, despite 
spike data confirming the neuron *was* firing.

**First validated the Monitor itself** by probing the input neuron (known 
behaviour from notebook 06: `vth=10, bias_mant=5` → voltage pattern 
`5, 10, 0` repeating). The Monitor reproduced this correctly, confirming 
the Monitor was working — the output neuron's voltage genuinely was zero 
at every sampled timestep.

**Hypothesis:** with weight=15 (well above threshold=10), the output neuron 
was crossing threshold and resetting within the same timestep the current 
arrived, every single step from step 3 onward — voltage was always back 
at 0 by the time it was sampled.

### Reducing synaptic weight to slow the dynamics
Reduced weight to 3 (requiring ~4 accumulated inputs to cross threshold=10) 
with `du=0` still active.

**Voltage trace (weight=3, du=0, no decay):**

| Step | Voltage |
|------|---------|
| 0 | 0 |
| 1 | 0 |
| 2 | 0 |
| 3 | 3 |
| 4 | 6 |
| 5 | 9 |
| 6 | 0 |
| 7 | 6 |
| 8 | 0 |
| 9 | 9 |
| 10 | 0 |
| 11 | 9 |
| 12 | 0 |
| 13 | 0 |
| 14 | 0 |
| 15 | 0 |
| 16 | 0 |
| 17 | 0 |
| 18 | 0 |
| 19 | 0 |

Output still fired far more often than the expected "1 spike per 4 input 
spikes" pattern. With `du=0`, every unit of current ever received remains 
fully active indefinitely and keeps compounding with new arriving current 
— current never decays, so contributions from old spikes never go away.

### Adding realistic current decay
Set `du=0.5` (current decays by half each step) and `dv=0.2` (slight voltage 
leak), better modelling a real biological post-synaptic current that rises 
and decays over time rather than delivering an instantaneous, permanent kick.

**Voltage trace (weight=3, du=0.5, dv=0.2):**

| Step | Voltage | Step | Voltage |
|------|---------|------|---------|
| 0 | 0.000 | 10 | 8.5030 |
| 1 | 0.000 | 11 | 7.6578 |
| 2 | 0.000 | 12 | 9.5540 |
| 3 | 3.000 | 13 | 9.3571 |
| 4 | 3.900 | 14 | 8.3426 |
| 5 | 3.870 | 15 | 0.000 |
| 6 | 6.471 | 16 | 1.7142 |
| 7 | 6.864 | 17 | 2.2285 |
| 8 | 6.335 | 18 | 5.2114 |
| 9 | 8.490 | 19 | 5.8834 |

This produced a single, clean output spike at step 15, with voltage rising 
and falling smoothly between input spikes — confirming current was decaying 
gradually rather than persisting indefinitely. The voltage rose even between 
new input spikes (e.g. step 3→4: 3.0→3.9) because previously-arrived current 
was still partially active and decaying, contributing to ongoing voltage rise.

### Key finding: the role of `du` (current decay)
`du=0` does not mean "no synaptic current" — it means **current never 
decays once delivered**, so every past spike's contribution remains 
permanently active and stacks with all future input. This produces runaway, 
continuous firing once enough accumulated current crosses threshold, 
regardless of input spike spacing.

Setting `du` to a nonzero value models a realistic decaying post-synaptic 
current, producing interpretable, biologically plausible accumulation and 
firing behaviour.

### Significance
This investigation demonstrates why explicit parameter understanding matters 
in neuromorphic frameworks — an "obvious" default (`du=0`) produced 
unintuitive runaway behaviour that was only properly understood through 
direct voltage probing rather than relying on the spike output alone. This 
same careful characterisation approach is exactly what's required when 
configuring real circuits on Loihi 2 hardware, where parameters like `du` 
and `dv` are physical circuit properties, not abstractions.